# Устанавливаем необходимые зависимости:

In [2]:
!pip install -U bitsandbytes accelerate transformers trl==0.19.0 datasets peft

  Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.19.0-py3-none-any.whl.metadata (10 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached torch-2.11.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install gdown


   ------------------------------ --------- 3/4 [gdown]
   ---------------------------------------- 4/4 [gdown]




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Обучение маленьких моделей

In [ ]:
from manim_finetune.utils import data_utils
from manim_finetune import constants
from manim_finetune.utils import model_utils
from manim_finetune.utils import archive_utils

Загружаем [данные](https://drive.google.com/file/d/1gb8S-8xharo5YYTOczAB_6YdOiQR_YCj/view?usp=drive_link):

In [ ]:
data_filename = "english_data.csv"
data_utils.download_data(constants.DATASET_DRIVE_ID, data_filename)
data = data_utils.load_data(data_filename)

Загружаем начальную модель, которую будем обучать (в нашем случае Qwen2.5-Coder-3B-Instruct)

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

In [ ]:
default_tokenizer, default_model = model_utils.load_model_and_tokenizer(
    constants.MODEL_NAME,
    bnb_config
)

Работаем с данными:

In [ ]:
train_dataset, test_dataset = data_utils.create_datasets(
    data,
    default_tokenizer,
    constants.TEST_SIZE,
    constants.MAX_LENGTH
)

Настраиваем модель:

In [ ]:
model = model_utils.apply_lora(
    default_model,
    constants.LORA_R,
    constants.LORA_ALPHA
)

In [ ]:
from trl import SFTConfig
import torch

torch.cuda.empty_cache()

training_args = SFTConfig(
    bf16=False,
    fp16=False,

    output_dir=constants.OUTPUT_DIR,
    num_train_epochs=constants.EPOCHS,
    per_device_train_batch_size=constants.BATCH_SIZE,
    gradient_accumulation_steps=constants.GRAD_ACCUM_STEPS,
    learning_rate=constants.LEARNING_RATE,

    dataset_text_field="text",

    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    save_steps=250,
    save_total_limit=5,

    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    prediction_loss_only=True,

    max_length=constants.MAX_LENGTH
)

In [ ]:
trainer = model_utils.create_trainer(model, default_tokenizer, train_dataset, test_dataset, training_args)

Обучаем:

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.763070,0.664380
100,0.686139,0.617855
150,0.635310,0.592750
200,0.632104,0.583900


Step,Training Loss,Validation Loss
50,0.763070,0.664380
100,0.686139,0.617855
150,0.635310,0.592750
200,0.632104,0.583900
250,0.567950,0.565285
300,0.562895,0.556277
350,0.522712,0.551810
400,0.528224,0.548864
450,0.495459,0.549641
500,0.460330,0.545810


TrainOutput(global_step=644, training_loss=0.5607716607751313, metrics={'train_runtime': 9000.1497, 'train_samples_per_second': 0.568, 'train_steps_per_second': 0.072, 'total_flos': 2.208213464174592e+16, 'train_loss': 0.5607716607751313})

Сохраняем дообученную модель:

In [ ]:
model_utils.save_model(trainer, constants.SAVE_MODEL_PATH)

In [ ]:
from manim_finetune.utils import archive_utils

archive_utils.make_zip(constants.SAVE_MODEL_PATH, constants.ZIP_NAME + ".zip")

# Оцениваем крутость модели

In [10]:
from manim_finetune.utils import archive_utils
from manim_finetune.utils import eval_utils
from manim_finetune import constants
from manim_finetune.utils import data_utils

Загружаем [модель](https://drive.google.com/file/d/1TgsqiUSQ6k_Wuw0LR2On1r5j765DeHgZ/view?usp=drive_link):

In [ ]:
tune_model_filename = constants.ZIP_NAME + ".zip"
data_utils.download_data(constants.TUNE_MODEL_DRIVE_ID, tune_model_filename)
archive_utils.extract_zip(tune_model_filename, constants.SAVE_MODEL_PATH)

Downloading...
From: https://drive.google.com/uc?id=1TgsqiUSQ6k_Wuw0LR2On1r5j765DeHgZ&confirm=t
To: c:\Users\User\Documents\Python Projects\Prompt2Scene\tune_model_english_qwen_coder.zip
100%|██████████| 7.28M/7.28M [00:05<00:00, 1.36MB/s]


In [ ]:
test_tokenizer, test_model = model_utils.load_model_and_tokenizer(constants.SAVE_MODEL_PATH)

Скачиваем данные:

In [ ]:
data_filename = "english_data.csv"
data_utils.download_data(constants.DATASET_DRIVE_ID, data_filename)
data = data_utils.load_data(data_filename)

Устанавливаем библиотеки для работы с manim:

In [ ]:
!sudo apt update
!sudo apt install libcairo2-dev \
    texlive texlive-latex-extra texlive-fonts-extra \
    texlive-latex-recommended texlive-science \
    tipa libpango1.0-dev
!pip install manim
!pip install IPython==8.21.0

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
132 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

In [ ]:
from manim_finetune.utils import gemini_utils
from manim_finetune import config

gemini_client = gemini_utils.get_gemini(config.GEMINI_API_KEY)

In [ ]:
from manim_finetune.utils import eval_utils
from manim_finetune.utils import manim_test_utils
from manim_finetune.utils import gemini_utils
from tqdm import tqdm

NUM_SAMPLES = 120
sample_df = data.sample(n=NUM_SAMPLES)

failed = list()
scores = dict()

for idx, row in tqdm(sample_df.iterrows(), total=NUM_SAMPLES):
    prompt = row['prompt']
    code = eval_utils.run_inference(test_model, test_tokenizer, prompt, constants.TEMPERATURE)
    if manim_test_utils.manim_test(code, idx=idx):
        score = gemini_utils.evaluate_code_with_gemini(gemini_client, prompt, code, constants.GEMINI_MODEL)
    else:
        score = 0
        failed.append(idx)
    scores[idx] = score

In [ ]:
print(f"Среднее (с учетом неудачных запусков): {sum(scores.values()) / len(scores)}")
print(f"Среднее (без учета неудачных запусков): {sum(scores.values()) / (len(scores) - len(failed))}")
print(f"Число неудачных запусков: {len(failed)}")
print(f"Неудачные запуски: {', '.join(failed)}")

# Обучение с подкреплением

In [6]:
!pip install google-genai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from manim_finetune.utils import gemini_utils
from manim_finetune import config

gemini_client = gemini_utils.get_gemini(config.GEMINI_API_KEY)

Загружаем модель из архива:

In [11]:
from manim_finetune import constants
from manim_finetune.utils import data_utils
from manim_finetune.utils import archive_utils

tune_model_filename = constants.ZIP_NAME + ".zip"
data_utils.download_data(constants.TUNE_MODEL_DRIVE_ID, tune_model_filename)
archive_utils.extract_zip(tune_model_filename, constants.SAVE_MODEL_PATH)

Downloading...
From: https://drive.google.com/uc?id=1TgsqiUSQ6k_Wuw0LR2On1r5j765DeHgZ&confirm=t
To: c:\Users\User\Documents\Python Projects\Prompt2Scene\tune_model_english_qwen_coder.zip
100%|██████████| 7.28M/7.28M [00:01<00:00, 3.80MB/s]


In [ ]:
test_tokenizer, test_model = model_utils.load_model_and_tokenizer(constants.SAVE_MODEL_PATH, False)

Скачиваем данные:

In [ ]:
from manim_finetune.utils import data_utils

data_filename = "english_data.csv"
data_utils.download_data(constants.DATASET_DRIVE_ID, data_filename)
data = data_utils.load_data(data_filename)

Генерируем пары "плохой-хороший" код для последующего обучения с подкреплением. В качестве судьи для оченки кода мы использовали `Gemma` (это оправдано тем, что у этих моделей щедрые ограничения)

In [ ]:
from manim_finetune.utils import rl_utils
from tqdm import tqdm

NUM_SAMPLES = 120
CANDIDATES_PER_PROMPT = 5

dpo_data = []

sample_df = data.sample(n=NUM_SAMPLES)

for idx, row in tqdm(sample_df.iterrows(), total=NUM_SAMPLES):
    prompt = row['prompt']
    pair = rl_utils.process_step(
        prompt,
        test_model,
        test_tokenizer,
        gemini_client,
        CANDIDATES_PER_PROMPT,
        constants.TEMPERATURE
    )
    if pair is not None:
        dpo_data.append(pair)

print(f"Collected {len(dpo_data)} pairs for DPO")

Сохраняем полученные пары в .csv файл:

In [ ]:
import pandas as pd
from manim_finetune.utils import data_utils
from manim_finetune import constants

dpo_dataset = pd.DataFrame(dpo_data)
data_utils.save_dataset(dpo_dataset, "dpo_preferences.csv")

Из-за технических ограничений нам пришлось перезапускать код выше несколько раз, для генерации хоть какого-то приличного количества пар. Все полученные датасеты (если Вам вдруг интересно) можно посмотреть на [Google Диске](https://drive.google.com/drive/folders/1s5zS-Hfu3quRkUKjJ4VMvCD8etfNNpBg?usp=drive_link).

Ниже загружается [датасет](https://drive.google.com/file/d/1QfRb-1mLdppG1zOpNpzgSiue3jV3xl22/view?usp=drive_link), который получен объединением всех датасетов из папки, которая была упомянута выше.

In [ ]:
from manim_finetune.utils import data_utils
from manim_finetune import constants

dpo_filename = "all_dpo_preferences.csv"
data_utils.download_data(constants.DPO_DATASET_DRIVE_ID, dpo_filename)
dpo_preferences = data_utils.load_data(dpo_filename)

c:\Users\User\Documents\Python Projects\Prompt2Scene\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Downloading...
From: https://drive.google.com/uc?id=1QfRb-1mLdppG1zOpNpzgSiue3jV3xl22&confirm=t
To: c:\Users\User\Documents\Python Projects\Prompt2Scene\dpo_preferences.csv
100%|██████████| 159k/159k [00:00<00:00, 2.36MB/s]


In [ ]:
from datasets import Dataset

# dpo_preferences = dpo_preferences[dpo_preferences["rejected_score"] == -1]
dpo_dataset = Dataset.from_pandas(dpo_preferences[["prompt", "chosen", "rejected"]])

Теперь дообучим модель на собранных парах:

In [ ]:
from transformers import BitsAndBytesConfig

dpo_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

In [ ]:
dpo_tokenizer, dpo_model = model_utils.load_model_and_tokenizer(constants.SAVE_MODEL_PATH, dpo_bnb_config)

In [ ]:
if dpo_tokenizer.pad_token is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

In [ ]:
from peft import prepare_model_for_kbit_training

dpo_model = prepare_model_for_kbit_training(dpo_model)
dpo_model.gradient_checkpointing_enable()

In [ ]:
from manim_finetune.utils import model_utils

dpo_model = model_utils.apply_lora(
    dpo_model,
    constants.DPO_LORA_R,
    constants.DPO_LORA_ALPHA,
    ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

In [ ]:
dpo_dataset = dpo_dataset.map(lambda example: rl_utils.format_dpo_string(example, dpo_tokenizer))

In [ ]:
from trl import DPOConfig

dpo_training_args = DPOConfig(
    output_dir=constants.DPO_OUTPUT_DIR,
    num_train_epochs=constants.DPO_EPOCHS,
    per_device_train_batch_size=constants.DPO_BATCH_SIZE,
    gradient_accumulation_steps=constants.DPO_GRAD_ACCUM_STEPS,
    learning_rate=constants.DPO_LEARNING_RATE,
    beta=constants.DPO_BETA,
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    bf16=False,
    fp16=False,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    max_length=constants.DPO_MAX_LENGTH,
    report_to="none"
)

In [ ]:
dpo_trainer = rl_utils.create_dpo_trainer(
    dpo_model,
    dpo_tokenizer,
    dpo_training_args,
    dpo_dataset
)

In [ ]:
dpo_trainer.train()

Сохраняем модель:

In [ ]:
model_utils.save_model(dpo_trainer, constants.DPO_SAVE_MODEL_PATH)

In [ ]:
from manim_finetune.utils import archive_utils

archive_utils.make_zip(constants.SAVE_MODEL_PATH, constants.DPO_ZIP_NAME + ".zip")

# Оценка качества обучения с подкреплением

In [ ]:
# TODO: Загрузка модели и данных

In [ ]:
!sudo apt update
!sudo apt install libcairo2-dev \
    texlive texlive-latex-extra texlive-fonts-extra \
    texlive-latex-recommended texlive-science \
    tipa libpango1.0-dev
!pip install manim
!pip install IPython==8.21.0

In [ ]:
from manim_finetune.utils import gemini_utils
from manim_finetune import config

gemini_client = gemini_utils.get_gemini(config.GEMINI_API_KEY)

In [ ]:
from manim_finetune.utils import eval_utils
from manim_finetune.utils import manim_test_utils
from manim_finetune.utils import gemini_utils
from tqdm import tqdm

NUM_SAMPLES = 120
sample_df = data.sample(n=NUM_SAMPLES)

failed = list()
scores = dict()

for idx, row in tqdm(sample_df.iterrows(), total=NUM_SAMPLES):
    prompt = row['prompt']
    code = eval_utils.run_inference(test_model, test_tokenizer, prompt, constants.TEMPERATURE)
    if manim_test_utils.manim_test(code, idx=idx):
        score = gemini_utils.evaluate_code_with_gemini(gemini_client, prompt, code, constants.GEMINI_MODEL)
    else:
        score = 0
        failed.append(idx)
    scores[idx] = score

In [ ]:
print(f"Среднее (с учетом неудачных запусков): {sum(scores.values()) / len(scores)}")
print(f"Среднее (без учета неудачных запусков): {sum(scores.values()) / (len(scores) - len(failed))}")
print(f"Число неудачных запусков: {len(failed)}")
print(f"Неудачные запуски: {', '.join(failed)}")